In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/smmehedizaman/glenn-beck/2021-07-12_Best_of_The_Program.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2025-05-07_Best_of_the_Program.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2025-06-27_NYC_s_Zohran_Mamdani.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2021-12-17_Best_of_the_Program.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2021-11-18_Fauci_Has_Explaining.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2023-08-15_Best_of_the_Program.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2021-11-09_Best_of_The_Program.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2021-01-28_GameStop_Stocks_Expl.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2021-09-22_Best_of_The_Program.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2021-01-30_Ep_95___Harvard_Astr.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2024-02-08_Best_of_the_Program.txt
/kaggle/input/datasets/smmehedizaman/glenn-beck/2021-11-03_It_s_a_Good_Day_for.txt


In [8]:
import os
import re

INPUT_DIR     = '/kaggle/input/datasets/smmehedizaman/glenn-beck'
OUTPUT_DIR    = '/kaggle/working/processed_trans'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1) Any turn containing one of these substrings will be dropped
AD_KEYWORDS = [
    'shopify','amazon','mint','hello fresh',
    'netsuite','oracle','mintmobile','cnn.com',
    'onlyarticlesoncnn.com','homemade','lifelock',
    'conebank','combat safe','sleep number','autism', 'Bill Par', 'celsius', 'pre-born', 'real estate'  # add any others you spot
]

# 2) Patterns that mark the start of the show
INTRO_PATTERNS = [
    re.compile(r"^welcome to the podcast", re.IGNORECASE)
]

def clean_transcription(content: str) -> str:
    # A) split into speaker turns
    turns = re.findall(
        r"(SPEAKER_\d+:.*?)(?=(?:SPEAKER_\d+:)|$)",
        content, flags=re.DOTALL
    )
    if not turns:
        return ""

    # B) remove empty & ad turns, keep (raw_turn, utterance) pairs
    filtered = []
    for raw in turns:
        # strip the label
        utter = re.sub(r"^SPEAKER_\d+:\s*", "", raw).strip()
        if not utter:
            continue
        low = utter.lower()
        if any(kw in low for kw in AD_KEYWORDS):
            continue
        filtered.append((raw, utter))

    if not filtered:
        return ""

    # C) find the index of the true opener
    start_idx = None
    for i, (_raw, utter) in enumerate(filtered):
        for pat in INTRO_PATTERNS:
            if pat.search(utter):
                start_idx = i
                break
        if start_idx is not None:
            break

    # D) fallback: if no explicit opener found, start at first filtered turn
    if start_idx is None:
        start_idx = 0

    # E) reassemble from that point onward
    cleaned = [raw for (raw, _) in filtered[start_idx:]]
    return "".join(cleaned)

# Process all files
for fname in os.listdir(INPUT_DIR):
    if not fname.endswith('.txt'):
        continue
    in_path  = os.path.join(INPUT_DIR,  fname)
    out_path = os.path.join(OUTPUT_DIR, fname)
    with open(in_path,  'r', encoding='utf-8') as f:
        raw = f.read()
    cleaned = clean_transcription(raw)
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(cleaned)

print(f"Cleaned transcripts saved to: {OUTPUT_DIR}")

Cleaned transcripts saved to: /kaggle/working/processed_trans


In [9]:
import os

# Path to the processed directory
processed_directory = '/kaggle/working/processed_trans'

# List a few processed files to check
sample_files = os.listdir(processed_directory)[:5]  # Get the first 5 files

for file_name in sample_files:
    file_path = os.path.join(processed_directory, file_name)
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
        print(f"Content of {file_name}:\n")
        print(content[:1000])  # Print the first 1000 characters for inspection
        print("-" * 80)

Content of 2024-04-23_Best_of_the_Program.txt:

SPEAKER_02:  We got a great show for you today. We...
SPEAKER_02:  We take on furries. We take on the Palestinian craziness. We are in really dangerous times. The economy, also a preview of Tomorrow Night's 90-Minute Special on the election. And it is a must watch. It'll be free for 24 hours on Blaze TV.
SPEAKER_02:  But we urge you to subscribe to BlazeTV and join us tomorrow.
SPEAKER_02:  You can, if you're not a subscriber, use Secure 2024 at blazetv.com slash Glenn and get your subscription. This is not only the problem, but we're including the solution to that problem. And you are critical and it's not... This is bipartisan. This is not partisan. I don't care who you voted for.
SPEAKER_02:  we all should agree that the election has got to be secure and uh...
SPEAKER_02:  We have found something that was presented at the Supreme Court and now is in open records.
SPEAKER_02:  away to hack in and possibly change
SPEAKER_02:  the results

In [10]:
import os
import re
import pandas as pd

# Function to count speaker turns in a file
def count_speaker_turns(file_path):
    """
    Counts the number of speaker turns in a transcript file.
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    # Match all speaker turns (e.g., "SPEAKER_XX:")
    speaker_turns = re.findall(r"SPEAKER_\d{2}:", content)
    return len(speaker_turns)

# Function to parse, label speakers, and create Anderson-Guest pairs
def parse_label_and_create_pairs(file_path, date=None):
    data = []
    pairs = []
    
    # Read the transcript file
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
        
    # Extract speaker turns
    for line in lines:
        line = line.strip()
        match = re.match(r"(SPEAKER_\d{2}):\s+(.*)", line)
        if match:
            speaker, utterance = match.groups()
            data.append({"Speaker": speaker, "Utterance": utterance})

    if not data:
        return pd.DataFrame()  # Return an empty DataFrame if no data exists

    # Identify the first speaker as Anderson
    first_speaker = data[0]['Speaker']

    # Label speakers: 'Anderson' for the first speaker, 'Guest' for others
    for entry in data:
        entry['Labeled_Speaker'] = 'Glenn' if entry['Speaker'] == first_speaker else 'Guest'

    # Create pairs while maintaining chronological order
    i = 0
    while i < len(data):
        if data[i]['Labeled_Speaker'] == 'Glenn':
            # Collect consecutive Anderson turns
            anderson_turns = [data[i]['Utterance']]
            i += 1
            while i < len(data) and data[i]['Labeled_Speaker'] == 'Glenn':
                anderson_turns.append(data[i]['Utterance'])
                i += 1
            
            # Check if the next turn is a Guest turn
            if i < len(data) and data[i]['Labeled_Speaker'] == 'Guest':
                guest_turns = [data[i]['Utterance']]
                guest_speaker = data[i]['Speaker']  # Track the unique guest speaker
                i += 1
                
                # Skip other guest turns from different speakers until it's Anderson's turn
                while i < len(data) and data[i]['Labeled_Speaker'] == 'Guest' and data[i]['Speaker'] == guest_speaker:
                    guest_turns.append(data[i]['Utterance'])
                    i += 1
                
                # Add the Anderson-Guest pair to the list
                pairs.append({"Date": date, "File": os.path.basename(file_path), 
                              "Glenn": " ".join(anderson_turns), 
                              "Guest": " ".join(guest_turns)})
            else:
                # If no Guest follows, skip to the next turn
                continue
        else:
            # Skip Guest turns until Anderson appears again
            i += 1

    # Convert pairs to DataFrame
    pairs_df = pd.DataFrame(pairs)
    return pairs_df

# Process all files in the directory
directory_path = '/kaggle/working/processed_trans'
output_csv = '/kaggle/working/glenn_guest_pairs.csv'

all_pairs = []

for file_name in os.listdir(directory_path):
    if file_name.endswith('.txt'):
        file_path = os.path.join(directory_path, file_name)
        
        # Count speaker turns and skip files with less than 10 turns
        num_turns = count_speaker_turns(file_path)
        if num_turns < 25:
            print(f"Skipping {file_name} (only {num_turns} speaker turns).")
            continue
        
        # Extract date from filename if available (e.g., YYYY-MM-DD in the filename)
        date_match = re.search(r"\d{4}-\d{2}-\d{2}", file_name)
        date = date_match.group(0) if date_match else None
        
        # Parse the file and append results
        pairs_df = parse_label_and_create_pairs(file_path, date)
        if not pairs_df.empty:
            all_pairs.append(pairs_df)

# Combine all pairs into a single DataFrame
final_df = pd.concat(all_pairs, ignore_index=True)

# Save to CSV
final_df.to_csv(output_csv, index=False, encoding='utf-8')
print(f"Combined Glenn-Guest pairs saved to {output_csv}")

Combined Glenn-Guest pairs saved to /kaggle/working/glenn_guest_pairs.csv


In [10]:
pd.set_option('display.max_colwidth', 4000)

In [14]:
final_df.head(10)

,Date,File,Glenn,Guest
0,2023-03-02,2023-03-02_Best_of_The_Program.txt,"Back hour with Biggs, more with the stuff on Russia. That was scary. Is that getting scary? It is getting scary, yeah. I think we... I think there's an initial... I thought I think by a lot of people like why are we worrying about this it's over there and yeah and it just now that we're over a year into it and Russia is saying things like yeah we can't lose because if we do it would be civil war so you know yeah we'll probably get to nuclear weapons if we're gonna lose this thing it's really bad and then","And then Iran, I mean, you know Israel's gonna do something. Israel's gonna do something. What is that gonna do to the world? Because remember Iran is now an ally of China and Russia. That's really good. I mean, there's just so much that you can go on, including the ransomware that we talked about on today's program. There's just so much. We had Andy Biggs on. He's trying to stop the funding in Congress, but he's pretty much fighting alone. I urge you to listen to that. Steve Crack-Hour talks about the media. From the inside out, he's written a new book called Uncovered and he's got people on record. By name, he wouldn't take any anonymous sources. And he's got people in the New York Times going, yeah, we're crazy. Yeah, we've done some really crazy things. I don't know what's happening. We're destroying media. He's got him from everywhere, sources inside mainstream media, telling you what's really going on. It's an eye-opening book. We also have Daniel Horowitz on talking about something else you have to worry about now that treaty with the WHO, which will take your sovereignty away. If we have another pandemic, which should be coming in 100 years if that's a 100 year event. We have another pandemic. The WHO will take 20% of all of our medical resources and distribute them all around the world. And we have to live by whatever the WHO is saying. That's good. That's really good. That's what Joe Biden is trying to ink now with the WHO. All this is so much more on today's podcast. First, our sponsor is Relief Factor. Going about your daily life with pain... ...socks. I mean... Psox! And I know the feeling because I have had severe pain for years and never find the right way to get around it or get out of it or anything just to enjoy your life."
1,2023-03-02,2023-03-02_Best_of_The_Program.txt,"And when you're in pain, it really not only affects you, but the people around you, like the people that you work with, that I have to, you know, work with you every day for like a multiple decade. You sound this sounds strangely specific.",you view them.
2,2023-03-02,2023-03-02_Best_of_The_Program.txt,"Look, I just...","I think that really factor is really important. Once I began taking it, almost all of my pain, except the pain in my ass went away. It is possible to get out of pain. Try the three week quick starts, only 1995. Trylpack, hundreds of thousands of people have ordered relief factor, and about 70% of them go on to order more. It's relieffactor.com, relieffactor.com, or call 800, the number four relief, 804 relief. Relieffactor.com, feel the difference."
3,2023-03-02,2023-03-02_Best_of_The_Program.txt,You're listening too. The best of the men back program.,"All right, there is a new story that is out from memory. Memory is a great organization that translates all of the stuff on Saudi Arabian, Qatar and all Arabic speaking channels because they don't say the same things in Arabic that they will in English. But I want you to be very well aware of what is being said there. There is just an interview that happened on Arabia, it's a network from Saudi Arabia. And it was with Alexander Dugan. It just happened on the 23rd and I just got the translation for it. Dugan said, and all of this is a quote. This is a very dangerous war, since Russia cannot lose it. This will bring Russia to its end. Russia cannot say, fine, we'll give up the areas we've taken over. Such 

In [11]:
# If you only want those two columns in your new df:
new_df = final_df[['Glenn', 'Guest']].rename(
    columns={'Glenn': 'speaker1', 'Guest': 'speaker2'}
)

In [12]:
new_df

,speaker1,speaker2
0,We got a great show for you today. We... We ta...,Anything else on Blaze TV when you're doing yo...
1,people to watch good.,"Really? Yeah, because I mean the place TV has ..."
2,Yeah. We have a couple that we just like are y...,You can't do that thing. lame legs?
3,"Yeah, you know, when I think of you and your s...",Yeah. get that.
4,called Studez America available.,What? every week right here on this network.
...,...,...
108378,It's not something I wouldn't wish it on my yo...,That's something I wouldn't wish to- My plan w...
108379,do,Yeah. Yeah. Right.
108380,"So Good, just do the next rape thumb. Thanks.","Only God knows where this goes, only God knows."
108381,"Exactly. Thank you so much, Carrie. Best of luck.","Thank you, Glenn."


In [13]:
# Save the new DataFrame to CSV.
new_df.to_csv('glenn_guest_utterances.csv', index=False)